# ModelLens on a toy transformer

This notebook runs **ModelLens** on `examples/toy_transformer.py` (**random weights**) to validate the inspection surface.

Use `modellens/visualization/showfig` for Plotly rendering.


## 1. Setup / imports

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "modellens").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from IPython.display import HTML, display

torch.manual_seed(42)

from examples.toy_transformer import ToyTransformer
from modellens import ModelLens

from modellens.analysis.activation_patching import run_activation_patching
from modellens.analysis.attention import compute_attention_pattern_metrics, run_attention_analysis
from modellens.analysis.backward_trace import run_backward_trace
from modellens.analysis.embeddings import run_embeddings_analysis
from modellens.analysis.forward_trace import run_forward_trace, trace_token_position_norms
from modellens.analysis.logit_lens import run_logit_lens
from modellens.analysis.residual_stream import run_residual_analysis

from modellens.analysis.training_snapshots import SnapshotStore, TrainingSnapshot

from modellens.visualization import (
    compute_shape_trace,
    format_patching_summary_html,
    plot_attention_head_entropy,
    plot_attention_head_grid,
    plot_attention_heatmap,
    plot_embedding_norms,
    plot_embedding_similarity_heatmap,
    plot_forward_family_aggregate,
    plot_forward_trace_norms,
    plot_activation_norm_distribution_by_family,
    plot_forward_trace_top_n,
    plot_gradient_norm_distribution_by_family,
    plot_gradient_norm_family_aggregate,
    plot_gradient_norm_top_n,
    plot_last_token_hidden_norm,
    plot_logit_lens_confidence_panel,
    plot_logit_lens_evolution,
    plot_logit_lens_heatmap,
    plot_logit_lens_top_token_bars,
    plot_module_gradient_norms,
    plot_parameter_sunburst_or_bar,
    plot_patching_family_effect_recovery_heatmap,
    plot_patching_importance_bar,
    plot_patching_recovery_fraction,
    plot_residual_contributions,
    plot_residual_lines,
    plot_snapshot_metric,
    plot_shape_trace_table,
    shape_trace_mermaid,
    shape_trace_to_dataframe,
)
from modellens.visualization.plotly_display import showfig
from modellens.visualization.overview import model_info_markdown

print("Imports OK")


## 2. Model construction & input

In [ ]:
VOCAB = 100
HIDDEN = 64
HEADS = 4
LAYERS = 3

model = ToyTransformer(
    vocab_size=VOCAB,
    hidden_dim=HIDDEN,
    num_heads=HEADS,
    num_layers=LAYERS,
)
model.eval()

input_ids = torch.tensor([[4, 7, 11, 2, 9, 0, 8, 15, 3, 22, 1, 5]], dtype=torch.long)

lens = ModelLens(model, backend="pytorch")

print("input_ids:", input_ids.shape, "backend:", lens.adapter.type_of_adapter)


## 3. Model overview / architecture

In [ ]:

showfig(plot_parameter_sunburst_or_bar(lens.model, max_depth=2, title="Toy parameters by prefix"))
display(HTML(model_info_markdown(lens, model_name="ToyTransformer")))
print(lens.summary())


## 4. Shape trace / execution trace

In [ ]:

lens.clear()
rows = compute_shape_trace(lens, input_ids)
print(f"Captured {len(rows)} module outputs")
showfig(plot_shape_trace_table(rows[:45], title="Toy — shape trace (first 45 modules)"))
print("Mermaid (preview):")
print(shape_trace_mermaid(rows, max_nodes=14))
display(shape_trace_to_dataframe(rows).head(8))


## 5. Forward pass explorer

In [ ]:

lens.clear()
ftr = run_forward_trace(lens, input_ids, max_modules=80)

showfig(plot_forward_trace_norms(ftr, summary_field="norm_mean"))
try:
    showfig(plot_last_token_hidden_norm(ftr))
except Exception as e:
    print("last-token plot:", e)

# Distribution / aggregation views
showfig(plot_forward_family_aggregate(ftr, summary_field="norm_mean", agg="mean", title="Forward — mean norm_mean by family"))
showfig(plot_activation_norm_distribution_by_family(ftr, summary_field="norm_mean", title="Forward — norm_mean distribution by family"))
showfig(plot_forward_trace_top_n(ftr, summary_field="norm_mean", top_n=12, title="Forward — top-12 by norm_mean"))


## 6. Token-centric hidden norm (selected position)

In [ ]:

POS = 5
lens.clear()
tp = trace_token_position_norms(lens, input_ids, position=POS)
print("position:", POS, "layers with norms:", len(tp["norms_by_layer"]))
for k, v in list(tp["norms_by_layer"].items())[:8]:
    print("  ", k, v)


## 7. Attention explorer

In [ ]:

lens.clear()
attn = run_attention_analysis(lens, input_ids)

showfig(plot_attention_heatmap(attn, layer_index=0, head_index=0))

# New auxiliary attention view
showfig(plot_attention_head_entropy(attn, layer_index=0, max_heads=12))

# Multi-head grid for quick visual diversity
showfig(plot_attention_head_grid(attn, layer_index=0, max_heads=min(HEADS, 8)))

metrics = compute_attention_pattern_metrics(attn)
print("attention heuristic metrics:")
print(metrics)


## 8. Logit lens / representation evolution

In [ ]:

lens.clear()
ll = run_logit_lens(lens, input_ids, tokenizer=None, top_k=5)

showfig(plot_logit_lens_evolution(ll, rank_index=0))
showfig(plot_logit_lens_heatmap(ll, top_ranks=5))
showfig(plot_logit_lens_confidence_panel(ll))
showfig(plot_logit_lens_top_token_bars(ll, layer_index=-1))


## 9. Residual stream

In [ ]:

block_names = [f"blocks.{i}" for i in range(LAYERS)]
lens.clear()
res = run_residual_analysis(lens, input_ids, layer_names=block_names)

showfig(plot_residual_contributions(res, mode="relative"))
showfig(plot_residual_contributions(res, mode="delta"))
showfig(plot_residual_lines(res))


## 10. Embeddings

In [ ]:

lens.clear()
emb = run_embeddings_analysis(lens, input_ids)
showfig(plot_embedding_similarity_heatmap(emb))
showfig(plot_embedding_norms(emb))


## 11. Activation patching (causal intervention)

In [ ]:

# Clean vs corrupted (toy: just corrupt last id)
clean_ids = input_ids.clone()
corrupted_ids = input_ids.clone()
corrupted_ids[0, -1] = 99

lens.clear()
patch = run_activation_patching(lens, clean_ids, corrupted_ids, layer_names=None)

display(HTML(format_patching_summary_html(patch)))

# Main effect + recovery views
showfig(plot_patching_importance_bar(patch, use_normalized=True, display_mode="full"))
showfig(plot_patching_recovery_fraction(patch, display_mode="full"))

# Family-level summary (new)
showfig(plot_patching_family_effect_recovery_heatmap(patch, use_normalized=True))

# Readability helpers
showfig(plot_patching_importance_bar(patch, use_normalized=True, display_mode="top_n", top_n=12))
showfig(plot_patching_recovery_fraction(patch, display_mode="top_n", top_n=12))

showfig(plot_patching_importance_bar(patch, use_normalized=True, display_mode="family"))
showfig(plot_patching_recovery_fraction(patch, display_mode="family"))


## 12. Backward / gradient flow

In [ ]:


# Surrogate: mean logits
lens.clear()
b1 = run_backward_trace(lens, input_ids, loss_mode="logits_mean")
showfig(plot_module_gradient_norms(b1, title="Gradients (mean logits loss) — per-module prefix"))
showfig(plot_gradient_norm_distribution_by_family(b1, title="Gradients — distribution by family"))
showfig(plot_gradient_norm_family_aggregate(b1, agg="mean", title="Gradients — mean by family"))
showfig(plot_gradient_norm_top_n(b1, top_n=12, title="Gradients — top-12 prefixes"))

# Surrogate: CE on last token
last_id = int(input_ids[0, -1].item())
lens.clear()
b2 = run_backward_trace(lens, input_ids, loss_mode="last_token_ce", target_token_id=last_id)
showfig(plot_module_gradient_norms(b2, title="Gradients (CE on last token) — per-module prefix"))


## 13. Training snapshots (lightweight demo)

In [ ]:

store = SnapshotStore()
store.append(TrainingSnapshot(step=0, train_loss=2.5, metrics={"grad_norm_l2": 0.5}))
store.append(TrainingSnapshot(step=100, train_loss=1.8, metrics={"grad_norm_l2": 0.3}))
showfig(plot_snapshot_metric(store.to_list(), "train_loss", title="train_loss vs step"))
showfig(plot_snapshot_metric(store.to_list(), "grad_norm_l2", title="grad_norm_l2 vs step"))


---

**Note:** This toy transformer is untrained; plots validate wiring and interfaces, not meaningful linguistic behavior.
